In [ ]:
import os
import glob
import torch
import torchaudio
import numpy as np
import pandas as pd
import torch.nn as nn
import timm
from tqdm import tqdm

BASE = "/kaggle/input/competitions/birdclef-2026/"
TAXONOMY_CSV = os.path.join(BASE, "taxonomy.csv")
TEST_AUDIO_DIR = os.path.join(BASE, "test_soundscapes")

MODEL_PATH = "/kaggle/input/notebooks/sofiasampara/notebook83485b055c/best_birdclef_model.pth"
THRESHOLDS_PATH = "/kaggle/input/notebooks/sofiasampara/notebook83485b055c/optimal_thresholds.npy"

class Config:
    SR = 32000
    DURATION = 5
    MAX_LENGTH = SR * DURATION
    N_MELS = 128
    N_FFT = 1024
    HOP_LENGTH = 512

taxonomy_df = pd.read_csv(TAXONOMY_CSV)
CLASSES = taxonomy_df['primary_label'].unique().tolist()
NUM_CLASSES = len(CLASSES)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
def apply_soft_calibration(probs, thresholds):
    """
    Sharpens probabilities around the per-class F1-optimized threshold.
    Values above the threshold are scaled toward 1.
    Values below the threshold are scaled toward 0.
    """
    scaled = np.copy(probs)
    num_classes = probs.shape[1]
    
    for c in range(num_classes):
        t = thresholds[c]
        above = probs[:, c] > t
        
        scaled[above, c] = 0.5 + 0.5 * (probs[above, c] - t) / (1.0 - t + 1e-8)        
        scaled[~above, c] = 0.5 * probs[~above, c] / (t + 1e-8)
        
    return np.clip(scaled, 0.0, 1.0)

In [ ]:
if os.path.exists(THRESHOLDS_PATH):
    optimal_thresholds = np.load(THRESHOLDS_PATH)
    print("Loaded F1-optimized thresholds successfully.")
else:
    print(f"WARNING: {THRESHOLDS_PATH} not found. Defaulting to 0.5 thresholds.")
    optimal_thresholds = np.full(NUM_CLASSES, 0.5, dtype=np.float32)

In [ ]:
class BirdCLEFModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=234):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=False, num_classes=0)
        in_features = self.backbone.num_features
        self.head = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))

model = BirdCLEFModel(num_classes=NUM_CLASSES)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.to(device)
model.eval()

In [ ]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=Config.SR,
    n_fft=Config.N_FFT,
    hop_length=Config.HOP_LENGTH,
    n_mels=Config.N_MELS,
    f_min=50,
    f_max=14000
).to(device)

amp_to_db = torchaudio.transforms.AmplitudeToDB().to(device)

test_files = glob.glob(os.path.join(TEST_AUDIO_DIR, "*.ogg"))

if len(test_files) == 0:
    print("Test directory empty. Using train_soundscapes for dry run...")
    test_files = glob.glob(os.path.join(BASE, "train_soundscapes", "*.ogg"))[:2]

In [ ]:
all_row_ids = []
all_raw_probs = []

print(f"Processing {len(test_files)} files...")
with torch.no_grad():
    for file_path in tqdm(test_files):
        filename = os.path.basename(file_path)
        file_id = filename.replace('.ogg', '')
        
        waveform, sr = torchaudio.load(file_path)
        
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        waveform = waveform.to(device)
        
        chunk_length = Config.MAX_LENGTH
        num_chunks = waveform.shape[1] // chunk_length
        
        if num_chunks == 0:
            pad_len = chunk_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, pad_len))
            num_chunks = 1
            
        for i in range(num_chunks):
            start = i * chunk_length
            chunk = waveform[:, start:start + chunk_length]
            
            mel_spec = mel_transform(chunk)
            mel_spec = amp_to_db(mel_spec)
            
            mel_spec = (mel_spec - mel_spec.min()) / (mel_spec.max() - mel_spec.min() + 1e-6)
            mel_spec = mel_spec * 2 - 1
            
            mel_spec = mel_spec.unsqueeze(0).expand(-1, 3, -1, -1)
            
            logits = model(mel_spec)
            probs = torch.sigmoid(logits).cpu().numpy()[0]
            
            end_time = (i + 1) * 5
            row_id = f"{file_id}_{end_time}"
            
            all_row_ids.append(row_id)
            all_raw_probs.append(probs)

raw_probs_matrix = np.array(all_raw_probs)

print("Applying soft calibration to raw probabilities...")
calibrated_probs = apply_soft_calibration(raw_probs_matrix, optimal_thresholds)

submission_df = pd.DataFrame(calibrated_probs, columns=CLASSES)
submission_df.insert(0, 'row_id', all_row_ids)

submission_df.to_csv('submission.csv', index=False)
print("submission.csv successfully created!")